#Data Analyst Agent

**Domain:** Agentic AI / Automated Analytics

**Dataset:** Superstore Sales Dataset -- https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

**Skills:** AI Agents, Tool Calling, Agent Workflows, Data Analysis Automation, LLM Orchestration

**Tools:** Python, LangGraph, LangChain, Cerebras / Groq / Ollama (free tier, via shared call_llm()), Pandas, Matplotlib

---

### Problem Statement
A small business doesn't have a dedicated analyst. They want to point an AI agent at a raw sales CSV and get back a finished analysis: key metrics calculated, charts generated, and a written report -- with the agent deciding which steps to take, not a fixed script.


In [ ]:
!pip install -q kagglehub langgraph langchain-core langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.2 MB/s eta 0:00:00


### Setup

In [ ]:
import pandas as pd
import numpy as np
# ---- AI setup: same pattern for every project, so it's one thing to remember ----
# we try cerebras first (biggest free daily limit), then groq, then a local ollama model
# if all 3 are down for some reason we just return a message instead of crashing the whole script
import os
from getpass import getpass
import requests
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore")

# asking for api keys only if they are not already set, and using getpass so the key
# does not get printed on screen or saved inside the notebook by accident
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key: ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

cerebras_client = OpenAI(api_key=os.environ["CEREBRAS_API_KEY"], base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

# once a provider fails (e.g. quota used up), remembering that for the rest of this session so
# we don't spam the same "cerebras failed: 402" error 30 times in a loop -- skip straight to
# the next provider instead
_dead_providers = set()

def call_llm(prompt):
    # step 1: try cerebras, it has the biggest free tier (1M tokens/day)
    if "cerebras" not in _dead_providers:
        try:
            response = cerebras_client.chat.completions.create(
                model="gpt-oss-120b",
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print("cerebras failed, switching to groq for the rest of this run:", e)
            _dead_providers.add("cerebras")

    # step 2: cerebras down or key wrong, try groq next
    if "groq" not in _dead_providers:
        try:
            response = groq_client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print("groq failed, switching to ollama for the rest of this run:", e)
            _dead_providers.add("groq")

    # step 3: last resort, try a local ollama model if one happens to be running
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"

import matplotlib.pyplot as plt
from langchain_core.tools import tool
from langchain_core.language_models.chat_models import SimpleChatModel
from langchain_core.messages import AIMessage
from langgraph.prebuilt import create_react_agent
# ---- dataset loading helper: tries kaggle first, never crashes the notebook ----
# kagglehub comes pre-installed on google colab, but just in case it's missing
# (running locally, older colab image etc) we try to pip install it quietly first
try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "kagglehub", "--break-system-packages"], check=False)
    try:
        import kagglehub
    except ImportError:
        kagglehub = None  # still not available, load_dataset_safe below will just use fake data

# a lot of the kaggle datasets in this project series randomly fail to download
# (auth issues, dataset moved, rate limit, no internet on the runtime, kagglehub missing etc)
# so instead of letting the whole notebook die, we fall back to a small fake dataset
# that has the exact same columns, so every line of code below still works
def load_dataset_safe(kaggle_handle, csv_name, synthetic_fn, n_rows=1500):
    if kagglehub is None:
        print("kagglehub isn't available, using synthetic sample data instead so the rest of the code still runs")
        return synthetic_fn(n_rows)
    try:
        path = kagglehub.dataset_download(kaggle_handle)
        full_path = os.path.join(path, csv_name)

        # the exact filename/path we expect doesn't always match what kaggle actually gives us
        # (nested folders, versioned paths, slightly different casing) -- if our guess isn't
        # there, look for any CSV in the download instead of giving up and going to synthetic data
        if not os.path.exists(full_path):
            csv_candidates = []
            for root, _dirs, files in os.walk(path):
                for fname in files:
                    if fname.lower().endswith('.csv'):
                        csv_candidates.append(os.path.join(root, fname))
            if csv_candidates:
                csv_candidates.sort(key=lambda p: 0 if csv_name.lower() in p.lower() else 1)
                full_path = csv_candidates[0]
                print(f"expected file not found at that exact path, using closest match instead: {os.path.basename(full_path)}")
            else:
                raise FileNotFoundError(f"no CSV files found anywhere in the downloaded dataset at {path}")

        # not every kaggle CSV is UTF-8 -- try that first, then fall back to latin-1/ISO-8859-1
        # instead of treating any decode error as a total failure and going to synthetic data
        try:
            df = pd.read_csv(full_path)
        except UnicodeDecodeError:
            df = pd.read_csv(full_path, encoding='ISO-8859-1')

        print(f"loaded real dataset from kaggle, {len(df)} rows")
        return df
    except Exception as e:
        print("could not download from kaggle (", e, ") - using synthetic sample data instead so the rest of the code still runs")
        return synthetic_fn(n_rows)


def make_fake_superstore(n_rows=2000):
    np.random.seed(20)
    regions = ['East', 'West', 'Central', 'South']
    cats = ['Furniture', 'Office Supplies', 'Technology']
    sales = np.round(np.random.exponential(200, n_rows), 2)
    return pd.DataFrame({
        'Region': np.random.choice(regions, n_rows), 'Category': np.random.choice(cats, n_rows),
        'Sales': sales, 'Profit': np.round(sales * np.random.uniform(-0.1, 0.3, n_rows), 2),
    })

from langchain_openai import ChatOpenAI

# create_react_agent needs a model that supports real tool-calling (bind_tools), which our
# own call_llm() wrapper doesn't do -- groq's api is openai-compatible and DOES support tool
# calling, so we point langchain's ChatOpenAI straight at groq instead
llm = ChatOpenAI(model="openai/gpt-oss-120b", api_key=os.environ["GROQ_API_KEY"],
                  base_url="https://api.groq.com/openai/v1", temperature=0)

# --- TOOL FUNCTIONS THE AGENT CAN CALL ---
_state = {}  # simple in-memory store so tools can share the loaded dataframe

@tool
def load_dataset(csv_path: str) -> str:
    """Loads a CSV file into memory for analysis. Call this first."""
    df = load_dataset_safe('vivek468/superstore-dataset-final', 'Sample - Superstore.csv',
                            make_fake_superstore, n_rows=2000)
    if 'Sales' not in df.columns:
        df = make_fake_superstore(2000)
    _state['df'] = df
    return f"loaded {len(df)} rows and {len(df.columns)} columns: {list(df.columns)}"

@tool
def compute_kpis() -> str:
    """Computes core business KPIs (total sales, total profit, top region, top category) from the loaded dataset."""
    df = _state['df']
    kpis = {
        "total_sales": round(df['Sales'].sum(), 2), "total_profit": round(df['Profit'].sum(), 2),
        "top_region": df.groupby('Region')['Sales'].sum().idxmax(),
        "top_category": df.groupby('Category')['Sales'].sum().idxmax(),
        "avg_order_value": round(df['Sales'].mean(), 2),
    }
    _state['kpis'] = kpis
    return str(kpis)

@tool
def generate_sales_chart() -> str:
    """Generates and saves a bar chart of sales by region from the loaded dataset."""
    df = _state['df']
    sales_by_region = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
    plt.figure(figsize=(8, 5))
    sales_by_region.plot(kind='bar')
    plt.title('Sales by Region')
    plt.tight_layout()
    plt.savefig('agent_sales_by_region.png')
    plt.close()
    return "chart saved to agent_sales_by_region.png"

@tool
def write_report() -> str:
    """Writes a short business report summarizing the computed KPIs. Call this last."""
    kpis = _state.get('kpis', {})
    return f"report drafted using KPIs: {kpis}"

# --- BUILD & RUN THE AGENT ---
tools = [load_dataset, compute_kpis, generate_sales_chart, write_report]
agent = create_react_agent(llm, tools)

result = agent.invoke({
    "messages": [("user", "Load sales_data.csv, compute the KPIs, generate a sales-by-region chart, "
                           "and write a short business report summarizing the findings.")]
})
print(result["messages"][-1].content)

Enter your Cerebras API key: ··········
Enter your Groq API key: ··········


100%|██████████| 550k/550k [00:00<00:00, 62.3MB/s]

Extracting files...
loaded real dataset from kaggle, 9994 rows


**Business Report – Sales Performance Summary**

**Key Findings**

| KPI | Value |
|-----|-------|
| **Total Sales** | **$2,297,200.86** |
| **Total Profit** | **$286,397.02** |
| **Average Order Value** | **$229.86** |
| **Top Performing Region** | **West** |
| **Top Performing Category** | **Technology** |

**Insights**

1. **Strong Overall Revenue** – The dataset shows over **$2.3 M** in sales, indicating a healthy top line.  
2. **Profitability** – With a profit of **$286 K**, the profit margin sits around **12.5 %**, suggesting solid cost control but also room for improvement.  
3. **Regional Strength** – The **West** region leads in sales, making it a strategic focus for continued investment and targeted marketing.  
4. **Category Leadership** – **Technology** is the top‑selling product category, driving the majority of revenue. Expanding the technology portfolio or bundling related accessories could further boost sales.  
5. **Average Order Value** – At **$229.86**, the AOV is a